# 03 · Feature engineering

Country-agnostic features per candidate pair (`src/ber/features.py`):

| family | features |
|---|---|
| retrieval | combined TF-IDF cosine `ret`, its name / address contributions `ret_nm` / `ret_ad`, rank `rk` among the record's neighbours |
| **competition** | for the S2/S3 record and for the S1 entity, on the combined, name and address scores: number of candidates, score relative to the best, margin to the best *other* candidate, gap between best and second best; number of near-tied rivals |
| **ambiguity** | how many S1 businesses in the block share exactly this name / this address (a name match is only trustworthy when the name is rare — e.g. an address-less `white medical centre` with three same-named S1s) |
| name strings | rapidfuzz `ratio`, `partial_ratio`, `token_sort`, `token_set`, Jaro-Winkler, full-name token set, no-space ratio (domains), alias-aware best score, TF-IDF char cosine, token Jaccard / overlap, first-token & exact equality, length ratio, token counts |
| legal form | Jaccard of canonical legal-form sets (NaN if either side has none) |
| address strings | `ratio`, `partial`, `token_sort`, `token_set`, TF-IDF word cosine, Jaccard, containment (partial addresses), empty flags, token counts |
| numbers | numeric-token Jaccard / shared count / conflict, first-number equality, postcode & state agreement |
| source | S3 vs S2 (formats differ) |

No country one-hot: the test set has a country (France) that training never saw.

**Training subsample.** Features are only needed for the entities used to fit and validate the
model: a random, entity-disjoint `TRAIN_PCT`% (fit) + `VALID_PCT`% (validation) of S1 entities,
with *all* of their candidates. Competition features still see every rival, because they come
from the statistics accumulated over the full candidate set in notebook 02. This keeps the
training table at ~2 GB instead of ~25 GB with no loss of realism.
Output: `features/train/{country}-{k}.parquet` (resumable parts).

In [ ]:
import sys, time, gc
from pathlib import Path

# notebooks/ -> ../src holds the shared, importable pipeline package `ber`
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ber import config as C, io, pipeline as P

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
C.ensure_dirs()
print(C.describe())
try:
    import psutil
    print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB | CPUs: {psutil.cpu_count()}")
except ImportError:
    pass

> **Infrastructure & cost.** Built to run the *full* dataset on a free-tier-class EC2 instance
> (m7i-flex.large: 2 vCPU, 8 GB RAM + 4 GB swap, 30 GB gp3) — every step streams or chunks
> its data. Checkpoints go to the local EBS volume (which survives stop/start) and small,
> valuable artifacts (model, metadata, submission) are mirrored to S3; big intermediates are
> kept local to stay inside the S3 free tier (5 GB, ~2k PUTs/month). S3→EC2 transfer in the
> same region is free. Re-running a notebook skips stages whose checkpoint exists
> (set `FORCE = True` to recompute). Everything is CPU-bound; no GPU is used.

In [ ]:
FORCE = False
t0 = time.time()
parts = P.run_features("train", force=FORCE)
print(f"{len(parts)} parts in {time.time() - t0:.0f}s")

## Sanity checks: coverage and separation between matches and non-matches

In [ ]:
from ber import features as F
feat = P.sample_features("train", frac=0.05)   # 5% row sample: summaries without loading ~16M rows
cols = F.feature_columns(feat)
print(f"sampled pairs {len(feat):,} | positives {feat.label.sum():,} ({100 * feat.label.mean():.2f}%) | features {len(cols)}")
print("sampled pairs by role (1=train, 2=valid-A, 3=valid-B):", feat.role.value_counts().sort_index().to_dict())
print("note: train-role easy negatives are subsampled (weight column); validation roles keep every pair")
summary = pd.DataFrame({"null_%": feat[cols].isna().mean().mul(100).round(2),
                        "mean_match": feat.loc[feat.label == 1, cols].mean(),
                        "mean_nonmatch": feat.loc[feat.label == 0, cols].mean()})
summary["gap"] = (summary.mean_match - summary.mean_nonmatch).abs()
summary.sort_values("gap", ascending=False).round(3)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, c in zip(axes.ravel(), ["grp_r_margin", "ret", "grp_r_ntie", "freq_r_name", "addr_contain", "num_jacc", "name_tset", "grp_s1_rel"]):
    for lab, col in ((0, "tab:red"), (1, "tab:blue")):
        v = feat.loc[feat.label == lab, c].dropna()
        ax.hist(v.sample(min(len(v), 300_000), random_state=0), bins=50, alpha=0.5, density=True, color=col, label=f"label={lab}")
    ax.set_title(c); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
del feat; gc.collect()